# Evaluate trained upper PPO on all controllable scenarios

Runs one checkpoint on every controllable Jain scenario and saves clean CSV artifacts for demand distribution before/after, QoS, throughput, handovers, and safe-admission diagnostics.

In [ ]:
import os
from pathlib import Path

if not Path('train_upper_ppo_3gnb.py').exists():
    os.chdir(Path.cwd().parent)

import json
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

DEFAULT_MODEL_CANDIDATES = [
    Path('models/upper_ppo_direct_last_continue_other_scenarios/run_20260702_155028/upper_ppo_final_after_all_other_scenarios.zip'),
    Path('models/upper_ppo_direct_last/run_20260701_232103/upper_ppo_final.zip'),
    Path('models/upper_ppo_masked_expert_direct/run_20260703_131439/upper_ppo_best.zip'),
]

MODEL_PATH = next((p for p in DEFAULT_MODEL_CANDIDATES if p.exists()), DEFAULT_MODEL_CANDIDATES[0])
OUT_DIR = Path('results') / 'all_controllable_model_demand_qos_eval_training_reward'
EPISODES_PER_SCENARIO = 10
SEED = 7
RERUN_EVAL = True
USE_TRAINING_REWARD_CONFIG = True
EXPERT_BIAS_CSV = Path('results/upper_heuristic_3gnb_baseline/upper_heuristic_3gnb_scenario_summary.csv')

print('Repo:', Path.cwd())
print('Model:', MODEL_PATH)
print('Output:', OUT_DIR)
print('Training reward config:', USE_TRAINING_REWARD_CONFIG)
assert MODEL_PATH.exists(), MODEL_PATH

In [ ]:
if RERUN_EVAL:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        'run_trained_policy_eval.py',
        '--model-path', str(MODEL_PATH),
        '--out-dir', str(OUT_DIR),
        '--episodes-per-scenario', str(EPISODES_PER_SCENARIO),
        '--seed', str(SEED),
    ]
    if USE_TRAINING_REWARD_CONFIG:
        cmd.append('--training-reward-config')
    elif EXPERT_BIAS_CSV.exists():
        cmd += ['--expert-bias-csv', str(EXPERT_BIAS_CSV)]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)

trace_files = sorted(OUT_DIR.glob('*_trained_policy_trace.csv'))
assert trace_files, f'No trace CSV files found in {OUT_DIR}'
print(f'Found {len(trace_files)} trace files')
for path in trace_files:
    print('-', path.name)

In [ ]:
frames = []
for path in trace_files:
    df = pd.read_csv(path)
    if 'scenario_name' not in df.columns:
        df['scenario_name'] = path.name.replace('_trained_policy_trace.csv', '')
    df['trace_file'] = str(path)
    frames.append(df)

trace = pd.concat(frames, ignore_index=True)
combined_trace_path = OUT_DIR / 'combined_trace.csv'
trace.to_csv(combined_trace_path, index=False)

print(trace.shape)
print('Saved:', combined_trace_path)
display(trace[['scenario_name', 'episode', 'episode_step', 'reward', 'handover_count', 'network_demand_prb_start', 'network_demand_prb_end', 'network_throughput_mbps', 'network_delivery_ratio']].head())

In [ ]:
GNBS = [0, 1, 2]
SLICES = ['eMBB', 'URLLC', 'mMTC']

demand_rows = []
for _, row in trace.iterrows():
    base = {
        'scenario_name': row['scenario_name'],
        'episode': row.get('episode', np.nan),
        'episode_step': row.get('episode_step', np.nan),
        'step': row.get('step', np.nan),
        'reward': row.get('reward', np.nan),
        'handover_count': row.get('handover_count', np.nan),
    }
    for phase in ['start', 'end']:
        for gnb in GNBS:
            gnb_total = row.get(f'gnb_demand_prb_{phase}_g{gnb}', np.nan)
            for slice_type in SLICES:
                demand_rows.append({
                    **base,
                    'phase': phase,
                    'gnb_id': gnb,
                    'slice_type': slice_type,
                    'demand_prb': row.get(f'demand_prb_{phase}_g{gnb}_{slice_type}', np.nan),
                    'gnb_total_demand_prb': gnb_total,
                    'used_prb': row.get(f'used_prb_{phase}_g{gnb}_{slice_type}', np.nan),
                    'ue_count': row.get(f'ue_count_g{gnb}_{slice_type}', np.nan),
                    'sla': row.get(f'sla_g{gnb}_{slice_type}', np.nan),
                })

demand_long = pd.DataFrame(demand_rows)
demand_long_path = OUT_DIR / 'demand_distribution_before_after_long.csv'
demand_long.to_csv(demand_long_path, index=False)

demand_summary = (
    demand_long
    .groupby(['scenario_name', 'phase', 'gnb_id', 'slice_type'], as_index=False)
    .agg(
        demand_prb_mean=('demand_prb', 'mean'),
        demand_prb_std=('demand_prb', 'std'),
        used_prb_mean=('used_prb', 'mean'),
        ue_count_mean=('ue_count', 'mean'),
        sla_mean=('sla', 'mean'),
    )
)
demand_summary_path = OUT_DIR / 'demand_distribution_before_after_summary.csv'
demand_summary.to_csv(demand_summary_path, index=False)

print('Saved:', demand_long_path)
print('Saved:', demand_summary_path)
display(demand_summary.head(18))

In [ ]:
network_metric_cols = [
    'reward',
    'handover_count',
    'network_demand_prb_start',
    'network_demand_prb_end',
    'mean_gnb_demand_prb_start',
    'mean_gnb_demand_prb_end',
    'max_gnb_demand_prb_start',
    'max_gnb_demand_prb_end',
    'gnb_total_demand_load_std_start',
    'gnb_total_demand_load_std_end',
    'network_used_prb_start',
    'network_used_prb_end',
    'network_throughput_mbps',
    'network_offered_mbps',
    'network_delivery_ratio',
    'network_completed_delay_ms',
    'network_mean_hol_delay_ms',
    'network_max_hol_delay_ms',
    'network_queue_kbits',
    'network_drop_ratio',
    'network_packet_failure_ratio',
    'paper_cost_reward',
    'expert_bias_reward',
    'paper_load_std_penalty',
    'paper_excess_load_penalty',
    'paper_handover_penalty',
    'paper_pingpong_penalty',
    'global_contradictory_bias_penalty',
    'idle_slice_bias_penalty',
    'expert_bias_mse',
    'expert_bias_closeness',
    'expert_bias_active_entries',
    'expert_bias_active_fraction',
    'paper_demand_load_std',
    'paper_useful_load_std',
    'jain_demand_raw',
    'jain_fairness_raw',
]
network_metric_cols = [c for c in network_metric_cols if c in trace.columns]

network_summary = (
    trace.groupby('scenario_name', as_index=False)[network_metric_cols]
    .agg(['mean', 'std', 'min', 'max'])
)
network_summary.columns = ['scenario_name'] + [f'{metric}_{stat}' for metric, stat in network_summary.columns[1:]]
network_summary_path = OUT_DIR / 'qos_throughput_network_summary.csv'
network_summary.to_csv(network_summary_path, index=False)

episode_cols = ['scenario_name', 'episode'] + network_metric_cols
episode_summary = trace[episode_cols].groupby(['scenario_name', 'episode'], as_index=False).mean(numeric_only=True)
episode_summary_path = OUT_DIR / 'qos_throughput_episode_summary.csv'
episode_summary.to_csv(episode_summary_path, index=False)

print('Saved:', network_summary_path)
print('Saved:', episode_summary_path)
display(network_summary)

In [ ]:
slice_rows = []
for _, row in trace.iterrows():
    base = {
        'scenario_name': row['scenario_name'],
        'episode': row.get('episode', np.nan),
        'episode_step': row.get('episode_step', np.nan),
        'reward': row.get('reward', np.nan),
        'handover_count': row.get('handover_count', np.nan),
    }
    for slice_type in SLICES:
        slice_rows.append({
            **base,
            'slice_type': slice_type,
            'demand_prb_start': row.get(f'slice_demand_prb_start_{slice_type}', np.nan),
            'demand_prb_end': row.get(f'slice_demand_prb_end_{slice_type}', np.nan),
            'used_prb_start': row.get(f'slice_used_prb_start_{slice_type}', np.nan),
            'used_prb_end': row.get(f'slice_used_prb_end_{slice_type}', np.nan),
            'throughput_mbps': row.get(f'qos_slice_throughput_mbps_{slice_type}', np.nan),
            'offered_mbps': row.get(f'qos_slice_offered_mbps_{slice_type}', np.nan),
            'delivery_ratio': row.get(f'qos_slice_delivery_ratio_{slice_type}', np.nan),
            'completed_delay_ms': row.get(f'qos_slice_completed_delay_ms_{slice_type}', np.nan),
            'mean_hol_delay_ms': row.get(f'qos_slice_mean_hol_delay_ms_{slice_type}', np.nan),
            'queue_kbits': row.get(f'qos_slice_queue_kbits_{slice_type}', np.nan),
            'drop_ratio': row.get(f'qos_slice_drop_ratio_{slice_type}', np.nan),
            'packet_failure_ratio': row.get(f'qos_slice_packet_failure_ratio_{slice_type}', np.nan),
            'sinr_db': row.get(f'qos_slice_sinr_db_{slice_type}', np.nan),
            'rsrq_db': row.get(f'qos_slice_rsrq_db_{slice_type}', np.nan),
        })

slice_qos = pd.DataFrame(slice_rows)
slice_qos_path = OUT_DIR / 'qos_throughput_per_slice_long.csv'
slice_qos.to_csv(slice_qos_path, index=False)

slice_summary = (
    slice_qos
    .groupby(['scenario_name', 'slice_type'], as_index=False)
    .mean(numeric_only=True)
)
slice_summary_path = OUT_DIR / 'qos_throughput_per_slice_summary.csv'
slice_summary.to_csv(slice_summary_path, index=False)

print('Saved:', slice_qos_path)
print('Saved:', slice_summary_path)
display(slice_summary)

In [ ]:
# Per-active-UE before/after comparison.
# The trace stores QoS by gNB/slice, not individual UE id. SINR and delay are
# already group averages over active UEs; throughput is converted to per-UE Mbps.
ordered_trace = trace.sort_values(['scenario_name', 'episode', 'step']).copy()
first_rows = ordered_trace.groupby(['scenario_name', 'episode'], as_index=False).first()
last_rows = ordered_trace.groupby(['scenario_name', 'episode'], as_index=False).last()

before_after_rows = []
for phase, phase_df in [('before', first_rows), ('after_balanced', last_rows)]:
    for _, row in phase_df.iterrows():
        for gnb in GNBS:
            for slice_type in SLICES:
                ue_count = row.get(f'ue_count_g{gnb}_{slice_type}', np.nan)
                throughput_mbps = row.get(f'qos_throughput_mbps_g{gnb}_{slice_type}', np.nan)
                active_ues = float(ue_count) if pd.notna(ue_count) else 0.0
                before_after_rows.append({
                    'scenario_name': row['scenario_name'],
                    'episode': row.get('episode', np.nan),
                    'phase': phase,
                    'gnb_id': gnb,
                    'slice_type': slice_type,
                    'active_ue_count': active_ues,
                    'sinr_db_avg_per_active_ue': row.get(f'qos_sinr_db_g{gnb}_{slice_type}', np.nan),
                    'throughput_mbps_total': throughput_mbps,
                    'throughput_mbps_per_active_ue': throughput_mbps / active_ues if active_ues > 0 else np.nan,
                    'completed_delay_ms_avg': row.get(f'qos_completed_delay_ms_g{gnb}_{slice_type}', np.nan),
                    'mean_hol_delay_ms_avg': row.get(f'qos_mean_hol_delay_ms_g{gnb}_{slice_type}', np.nan),
                    'delivery_ratio': row.get(f'qos_delivery_ratio_g{gnb}_{slice_type}', np.nan),
                    'demand_prb': row.get(f'demand_prb_{"start" if phase == "before" else "end"}_g{gnb}_{slice_type}', np.nan),
                })

ue_qos_before_after = pd.DataFrame(before_after_rows)
ue_qos_before_after = ue_qos_before_after[ue_qos_before_after['active_ue_count'] > 0].copy()
ue_qos_path = OUT_DIR / 'per_active_ue_qos_before_after_long.csv'
ue_qos_before_after.to_csv(ue_qos_path, index=False)

ue_qos_summary = (
    ue_qos_before_after
    .groupby(['scenario_name', 'phase', 'gnb_id', 'slice_type'], as_index=False)
    .agg(
        active_ue_count_mean=('active_ue_count', 'mean'),
        sinr_db_avg_per_active_ue=('sinr_db_avg_per_active_ue', 'mean'),
        throughput_mbps_per_active_ue=('throughput_mbps_per_active_ue', 'mean'),
        throughput_mbps_total=('throughput_mbps_total', 'mean'),
        completed_delay_ms_avg=('completed_delay_ms_avg', 'mean'),
        mean_hol_delay_ms_avg=('mean_hol_delay_ms_avg', 'mean'),
        delivery_ratio=('delivery_ratio', 'mean'),
        demand_prb=('demand_prb', 'mean'),
    )
)
ue_qos_summary_path = OUT_DIR / 'per_active_ue_qos_before_after_summary.csv'
ue_qos_summary.to_csv(ue_qos_summary_path, index=False)

print('Saved:', ue_qos_path)
print('Saved:', ue_qos_summary_path)
display(ue_qos_summary.head(24))

In [ ]:
reward_component_cols = [
    'paper_cost_reward',
    'expert_bias_reward',
    'paper_load_std_penalty',
    'paper_excess_load_penalty',
    'paper_handover_penalty',
    'paper_pingpong_penalty',
    'global_contradictory_bias_penalty',
    'idle_slice_bias_penalty',
    'expert_bias_mse',
    'expert_bias_closeness',
    'expert_bias_active_entries',
    'expert_bias_active_fraction',
]
reward_component_cols = [c for c in reward_component_cols if c in trace.columns]

reward_components = (
    trace.groupby('scenario_name', as_index=False)[['reward'] + reward_component_cols]
    .mean(numeric_only=True)
)
reward_components_path = OUT_DIR / 'reward_components_summary.csv'
reward_components.to_csv(reward_components_path, index=False)

episode_returns = (
    trace.groupby(['scenario_name', 'episode'], as_index=False)
    .agg(
        episode_return=('reward', 'sum'),
        handovers=('handover_count', 'sum'),
        mean_sla_count=('sla_count', 'mean'),
        final_demand_std=('gnb_total_demand_load_std_end', 'mean'),
        mean_throughput_mbps=('network_throughput_mbps', 'mean'),
        mean_delivery_ratio=('network_delivery_ratio', 'mean'),
    )
)
episode_returns_path = OUT_DIR / 'episode_return_diagnostics.csv'
episode_returns.to_csv(episode_returns_path, index=False)

print('Saved:', reward_components_path)
print('Saved:', episode_returns_path)
display(reward_components)


In [ ]:
import matplotlib.pyplot as plt

plot_df = demand_summary.copy()
plot_df['gnb_slice'] = plot_df['gnb_id'].astype(str).radd('g') + ' ' + plot_df['slice_type']

scenarios = list(plot_df['scenario_name'].drop_duplicates())
fig, axes = plt.subplots(len(scenarios), 1, figsize=(14, max(3.2, 2.8 * len(scenarios))), sharex=True)
if len(scenarios) == 1:
    axes = [axes]

for ax, scenario in zip(axes, scenarios):
    sub = plot_df[plot_df['scenario_name'] == scenario]
    pivot = sub.pivot_table(index='gnb_slice', columns='phase', values='demand_prb_mean', aggfunc='mean').reindex(columns=['start', 'end'])
    pivot.plot(kind='bar', ax=ax, width=0.78)
    ax.set_title(scenario)
    ax.set_ylabel('Demand PRB')
    ax.grid(axis='y', alpha=0.25)

axes[-1].set_xlabel('gNB / slice')
plt.tight_layout()
demand_plot_path = OUT_DIR / 'demand_distribution_before_after.png'
plt.savefig(demand_plot_path, dpi=160, bbox_inches='tight')
print('Saved:', demand_plot_path)
plt.show()

In [ ]:
qos_plot_cols = [
    'network_throughput_mbps_mean',
    'network_delivery_ratio_mean',
    'network_mean_hol_delay_ms_mean',
    'handover_count_mean',
    'gnb_total_demand_load_std_end_mean',
]
qos_plot_cols = [c for c in qos_plot_cols if c in network_summary.columns]

fig, axes = plt.subplots(len(qos_plot_cols), 1, figsize=(13, max(3, 2.6 * len(qos_plot_cols))), sharex=True)
if len(qos_plot_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, qos_plot_cols):
    ax.bar(network_summary['scenario_name'], network_summary[col])
    ax.set_ylabel(col.replace('_mean', ''))
    ax.grid(axis='y', alpha=0.25)

axes[-1].tick_params(axis='x', rotation=35)
plt.tight_layout()
qos_plot_path = OUT_DIR / 'qos_throughput_summary.png'
plt.savefig(qos_plot_path, dpi=160, bbox_inches='tight')
print('Saved:', qos_plot_path)
plt.show()

In [ ]:
component_plot_cols = [
    'expert_bias_reward',
    'paper_load_std_penalty',
    'paper_excess_load_penalty',
    'paper_handover_penalty',
    'paper_pingpong_penalty',
    'global_contradictory_bias_penalty',
    'idle_slice_bias_penalty',
]
component_plot_cols = [c for c in component_plot_cols if c in reward_components.columns]
component_plot = reward_components[['scenario_name'] + component_plot_cols].set_index('scenario_name').copy()
for col in component_plot.columns:
    if col != 'expert_bias_reward':
        component_plot[col] = -component_plot[col]

ax = component_plot.plot(kind='bar', stacked=True, figsize=(14, 5.5), width=0.82)
ax.axhline(0, color='black', linewidth=0.9)
ax.set_ylabel('Mean reward contribution')
ax.grid(axis='y', alpha=0.25)
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
reward_component_plot_path = OUT_DIR / 'reward_components_stacked.png'
plt.savefig(reward_component_plot_path, dpi=160, bbox_inches='tight')
print('Saved:', reward_component_plot_path)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

axes[0].scatter(episode_returns['handovers'], episode_returns['episode_return'], s=58, alpha=0.8)
axes[0].set_xlabel('Episode handovers')
axes[0].set_ylabel('Episode return')
axes[0].grid(alpha=0.25)

axes[1].scatter(episode_returns['final_demand_std'], episode_returns['episode_return'], s=58, alpha=0.8)
axes[1].set_xlabel('Final demand-load std')
axes[1].set_ylabel('Episode return')
axes[1].grid(alpha=0.25)

axes[2].scatter(episode_returns['mean_delivery_ratio'], episode_returns['mean_throughput_mbps'], s=58, alpha=0.8)
axes[2].set_xlabel('Mean delivery ratio')
axes[2].set_ylabel('Mean throughput Mbps')
axes[2].grid(alpha=0.25)

plt.tight_layout()
diagnostic_scatter_path = OUT_DIR / 'episode_return_diagnostic_scatter.png'
plt.savefig(diagnostic_scatter_path, dpi=160, bbox_inches='tight')
print('Saved:', diagnostic_scatter_path)
plt.show()


In [ ]:
load_std_cols = [
    'gnb_total_demand_load_std_start_mean',
    'gnb_total_demand_load_std_end_mean',
    'paper_demand_load_std_mean',
]
load_std_cols = [c for c in load_std_cols if c in network_summary.columns]
load_std = network_summary[['scenario_name'] + load_std_cols].set_index('scenario_name')

ax = load_std.plot(kind='bar', figsize=(13, 4.8), width=0.78)
ax.set_ylabel('Demand-load std')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
load_std_plot_path = OUT_DIR / 'demand_load_std_before_after.png'
plt.savefig(load_std_plot_path, dpi=160, bbox_inches='tight')
print('Saved:', load_std_plot_path)
plt.show()


In [ ]:
slice_plot = slice_summary.copy()
slice_plot['scenario_slice'] = slice_plot['scenario_name'] + ' / ' + slice_plot['slice_type']
slice_plot = slice_plot.sort_values(['scenario_name', 'slice_type'])

fig, axes = plt.subplots(2, 1, figsize=(15, 7.5), sharex=True)
axes[0].bar(slice_plot['scenario_slice'], slice_plot['throughput_mbps'])
axes[0].set_ylabel('Throughput Mbps')
axes[0].grid(axis='y', alpha=0.25)

axes[1].bar(slice_plot['scenario_slice'], slice_plot['delivery_ratio'])
axes[1].set_ylabel('Delivery ratio')
axes[1].set_ylim(0, max(1.05, float(slice_plot['delivery_ratio'].max()) * 1.05 if len(slice_plot) else 1.05))
axes[1].grid(axis='y', alpha=0.25)
axes[1].tick_params(axis='x', rotation=60)

plt.tight_layout()
slice_qos_plot_path = OUT_DIR / 'per_slice_throughput_delivery.png'
plt.savefig(slice_qos_plot_path, dpi=160, bbox_inches='tight')
print('Saved:', slice_qos_plot_path)
plt.show()


In [ ]:
ue_metric_labels = {
    'sinr_db_avg_per_active_ue': 'SINR dB avg per active UE',
    'throughput_mbps_per_active_ue': 'Throughput Mbps per active UE',
    'completed_delay_ms_avg': 'Completed delay ms avg',
    'mean_hol_delay_ms_avg': 'HOL delay ms avg',
}

plot_base = ue_qos_summary.copy()
plot_base['gnb_slice'] = plot_base['gnb_id'].astype(str).radd('g') + ' ' + plot_base['slice_type']
plot_base['scenario_gnb_slice'] = plot_base['scenario_name'] + ' / ' + plot_base['gnb_slice']
print('Trace aggregate QoS note: these columns come from post-window CSV snapshots. If before/after is identical here, use the direct per-UE cells below.')

for metric, ylabel in ue_metric_labels.items():
    metric_df = plot_base.pivot_table(
        index='scenario_gnb_slice',
        columns='phase',
        values=metric,
        aggfunc='mean',
    ).reindex(columns=['before', 'after_balanced']).dropna(how='all')
    if metric_df.empty:
        continue
    if {'before', 'after_balanced'} <= set(metric_df.columns):
        max_delta = (metric_df['after_balanced'] - metric_df['before']).abs().max()
        if pd.isna(max_delta) or max_delta < 1e-9:
            print(f'Skipping {metric}: trace aggregate before/after is identical; direct per-UE snapshot below is the real comparison.')
            continue
    ax = metric_df.plot(kind='bar', figsize=(16, 5.4), width=0.78)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=65)
    plt.tight_layout()
    metric_plot_path = OUT_DIR / f'per_active_ue_{metric}_before_after.png'
    plt.savefig(metric_plot_path, dpi=160, bbox_inches='tight')
    print('Saved:', metric_plot_path)
    plt.show()


In [ ]:
delta_rows = []
key_cols = ['scenario_name', 'gnb_id', 'slice_type']
for metric in ue_metric_labels:
    wide = ue_qos_summary.pivot_table(index=key_cols, columns='phase', values=metric, aggfunc='mean')
    if {'before', 'after_balanced'} <= set(wide.columns):
        delta = (wide['after_balanced'] - wide['before']).rename(metric + '_delta').reset_index()
        delta_rows.append(delta)

if delta_rows:
    ue_qos_delta = delta_rows[0]
    for frame in delta_rows[1:]:
        ue_qos_delta = ue_qos_delta.merge(frame, on=key_cols, how='outer')
else:
    ue_qos_delta = pd.DataFrame(columns=key_cols)

ue_qos_delta_path = OUT_DIR / 'per_active_ue_qos_before_after_delta.csv'
ue_qos_delta.to_csv(ue_qos_delta_path, index=False)
print('Saved:', ue_qos_delta_path)
display(ue_qos_delta)

In [ ]:
# Direct per-UE radio snapshot: before policy action vs after the balancing step.
# This reruns one deterministic episode per scenario because the CSV trace stores
# per-gNB/slice QoS, while this cell needs individual UE radio metrics.
from types import SimpleNamespace
from stable_baselines3 import PPO

from run_zero_action_baseline import CONTROLLABLE_SCENARIOS, build_args
from run_trained_policy_eval import apply_reward_overrides
from train_upper_ppo_3gnb import make_env

RUN_DIRECT_PER_UE_SNAPSHOT = True
PER_UE_DETAIL_SCENARIO = 'jain_control_mixed'

def training_reward_override_args():
    return SimpleNamespace(
        training_reward_config=USE_TRAINING_REWARD_CONFIG,
        paper_handover_penalty_weight=None,
        paper_pingpong_penalty_weight=None,
        expert_bias_reward_weight=None,
        expert_bias_closeness_threshold=None,
        contradictory_bias_penalty_weight=None,
        idle_slice_bias_penalty_weight=None,
        expert_bias_csv=None,
        no_expert_bias_csv=False,
    )

def capture_per_ue_rows(env, scenario_name, phase):
    base_env = env.unwrapped.base_env
    rows = []
    for ue in base_env.get_all_ues():
        metrics = base_env.get_ue_radio_metrics(int(ue.id))
        rows.append({
            'scenario_name': scenario_name,
            'phase': phase,
            'ue_id': int(metrics.get('ue_id', ue.id)),
            'slice_type': getattr(ue, 'slice_type', metrics.get('slice_type', '')),
            'serving_gnb': metrics.get('serving_gnb'),
            'connected': bool(metrics.get('connected', False)),
            'x': metrics.get('x', np.nan),
            'y': metrics.get('y', np.nan),
            'sinr_db': metrics.get('sinr_db', np.nan),
            'scheduled_sinr_db': metrics.get('scheduled_sinr_db', np.nan),
            'rsrq_db': metrics.get('rsrq_db', np.nan),
            'throughput_bps': metrics.get('throughput', np.nan),
            'throughput_mbps': metrics.get('throughput', np.nan) / 1e6,
            'delay_steps': metrics.get('delay_steps', np.nan),
            'queue_bits': metrics.get('queue', np.nan),
            'allocated_prbs': metrics.get('allocated_prbs', np.nan),
            'used_prbs': metrics.get('used_prbs', np.nan),
            'useful_prbs': metrics.get('useful_prbs', np.nan),
            'rx_probability': metrics.get('rx_probability', np.nan),
            'mcs': metrics.get('mcs', np.nan),
            'target_gnb': metrics.get('target_gnb', np.nan),
            'ho_pending': bool(metrics.get('ho_pending', False)),
            'ho_candidate': metrics.get('ho_candidate', np.nan),
        })
    return rows

if RUN_DIRECT_PER_UE_SNAPSHOT:
    model = PPO.load(str(MODEL_PATH), device='cpu')
    per_ue_rows = []
    for scenario_name in CONTROLLABLE_SCENARIOS:
        env_args = apply_reward_overrides(build_args(scenario_name, SEED), training_reward_override_args())
        env = make_env(env_args)
        obs, reset_info = env.reset()
        per_ue_rows.extend(capture_per_ue_rows(env, scenario_name, 'before'))
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        after_rows = capture_per_ue_rows(env, scenario_name, 'after_balanced')
        for row in after_rows:
            row['reward'] = float(reward)
            row['handover_count'] = int(info.get('handover_count', 0))
        per_ue_rows.extend(after_rows)
        env.close()

    per_ue_radio = pd.DataFrame(per_ue_rows)
else:
    per_ue_radio = pd.read_csv(OUT_DIR / 'per_ue_radio_before_after_long.csv')

per_ue_radio_path = OUT_DIR / 'per_ue_radio_before_after_long.csv'
per_ue_summary = (
    per_ue_radio
    .groupby(['scenario_name', 'phase'], as_index=False)
    .agg(
        ue_count=('ue_id', 'count'),
        sinr_db=('sinr_db', 'mean'),
        scheduled_sinr_db=('scheduled_sinr_db', 'mean'),
        throughput_mbps=('throughput_mbps', 'mean'),
        delay_steps=('delay_steps', 'mean'),
        queue_bits=('queue_bits', 'mean'),
        useful_prbs=('useful_prbs', 'mean'),
        rx_probability=('rx_probability', 'mean'),
    )
)
per_ue_summary_path = OUT_DIR / 'per_ue_radio_before_after_summary.csv'
per_ue_summary.to_csv(per_ue_summary_path, index=False)

serving_wide = per_ue_radio.pivot_table(
    index=['scenario_name', 'ue_id'],
    columns='phase',
    values='serving_gnb',
    aggfunc='first',
)
if {'before', 'after_balanced'} <= set(serving_wide.columns):
    moved_index = serving_wide[serving_wide['before'] != serving_wide['after_balanced']].index
    moved_keys = pd.DataFrame(list(moved_index), columns=['scenario_name', 'ue_id'])
    moved_keys['moved'] = True
    per_ue_radio = per_ue_radio.merge(moved_keys, on=['scenario_name', 'ue_id'], how='left')
    per_ue_radio['moved'] = per_ue_radio['moved'].fillna(False)
else:
    per_ue_radio['moved'] = False
per_ue_radio.to_csv(per_ue_radio_path, index=False)

moved_ue_radio = per_ue_radio[per_ue_radio['moved']].copy()
moved_ue_path = OUT_DIR / 'moved_ue_radio_before_after_long.csv'
moved_ue_radio.to_csv(moved_ue_path, index=False)

print('Saved:', per_ue_radio_path)
print('Saved:', per_ue_summary_path)
print('Saved:', moved_ue_path)
print('Moved UEs:', moved_ue_radio[['scenario_name', 'ue_id']].drop_duplicates().shape[0])
display(per_ue_radio.head(12))
display(per_ue_summary)

In [ ]:
per_ue_metrics = {
    'sinr_db': 'Mean per-UE SINR dB',
    'throughput_mbps': 'Mean per-UE throughput Mbps',
    'delay_steps': 'Mean per-UE delay steps',
    'queue_bits': 'Mean per-UE queue bits',
}

fig, axes = plt.subplots(len(per_ue_metrics), 1, figsize=(14, 3.4 * len(per_ue_metrics)), sharex=True)
if len(per_ue_metrics) == 1:
    axes = [axes]

for ax, (metric, ylabel) in zip(axes, per_ue_metrics.items()):
    pivot = per_ue_summary.pivot_table(index='scenario_name', columns='phase', values=metric, aggfunc='mean').reindex(columns=['before', 'after_balanced'])
    pivot.plot(kind='bar', ax=ax, width=0.78)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.25)

axes[-1].tick_params(axis='x', rotation=35)
plt.tight_layout()
per_ue_summary_plot_path = OUT_DIR / 'per_ue_radio_before_after_summary.png'
plt.savefig(per_ue_summary_plot_path, dpi=160, bbox_inches='tight')
print('Saved:', per_ue_summary_plot_path)
plt.show()

detail = per_ue_radio[per_ue_radio['scenario_name'] == PER_UE_DETAIL_SCENARIO].copy()
detail['ue_label'] = detail['slice_type'].astype(str) + ' ue' + detail['ue_id'].astype(str)
detail_metrics = ['sinr_db', 'throughput_mbps', 'delay_steps']
fig, axes = plt.subplots(len(detail_metrics), 1, figsize=(15, 8.5), sharex=True)
for ax, metric in zip(axes, detail_metrics):
    pivot = detail.pivot_table(index='ue_label', columns='phase', values=metric, aggfunc='mean').reindex(columns=['before', 'after_balanced'])
    pivot.plot(kind='bar', ax=ax, width=0.78)
    ax.set_ylabel(metric)
    ax.grid(axis='y', alpha=0.25)
axes[-1].tick_params(axis='x', rotation=65)
plt.tight_layout()
per_ue_detail_plot_path = OUT_DIR / f'per_ue_radio_before_after_{PER_UE_DETAIL_SCENARIO}.png'
plt.savefig(per_ue_detail_plot_path, dpi=160, bbox_inches='tight')
print('Saved:', per_ue_detail_plot_path)
plt.show()

In [ ]:
if moved_ue_radio.empty:
    print('No moved UEs found in the direct snapshot.')
else:
    moved_summary = (
        moved_ue_radio
        .groupby(['scenario_name', 'phase'], as_index=False)
        .agg(
            moved_ue_count=('ue_id', 'nunique'),
            sinr_db=('sinr_db', 'mean'),
            throughput_mbps=('throughput_mbps', 'mean'),
            delay_steps=('delay_steps', 'mean'),
            queue_bits=('queue_bits', 'mean'),
            useful_prbs=('useful_prbs', 'mean'),
        )
    )
    moved_summary_path = OUT_DIR / 'moved_ue_radio_before_after_summary.csv'
    moved_summary.to_csv(moved_summary_path, index=False)
    print('Saved:', moved_summary_path)
    display(moved_summary)

    moved_metrics = {
        'sinr_db': 'Moved UE SINR dB',
        'throughput_mbps': 'Moved UE throughput Mbps',
        'delay_steps': 'Moved UE delay steps',
    }
    fig, axes = plt.subplots(len(moved_metrics), 1, figsize=(14, 3.5 * len(moved_metrics)), sharex=True)
    if len(moved_metrics) == 1:
        axes = [axes]
    for ax, (metric, ylabel) in zip(axes, moved_metrics.items()):
        pivot = moved_summary.pivot_table(index='scenario_name', columns='phase', values=metric, aggfunc='mean').reindex(columns=['before', 'after_balanced'])
        pivot.plot(kind='bar', ax=ax, width=0.78)
        ax.set_ylabel(ylabel)
        ax.grid(axis='y', alpha=0.25)
    axes[-1].tick_params(axis='x', rotation=35)
    plt.tight_layout()
    moved_summary_plot_path = OUT_DIR / 'moved_ue_radio_before_after_summary.png'
    plt.savefig(moved_summary_plot_path, dpi=160, bbox_inches='tight')
    print('Saved:', moved_summary_plot_path)
    plt.show()

    detail = moved_ue_radio[moved_ue_radio['scenario_name'] == PER_UE_DETAIL_SCENARIO].copy()
    if detail.empty:
        detail = moved_ue_radio.copy()
    detail['ue_label'] = detail['scenario_name'] + ' / ' + detail['slice_type'].astype(str) + ' ue' + detail['ue_id'].astype(str)
    fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True)
    for ax, metric in zip(axes, ['sinr_db', 'throughput_mbps', 'delay_steps']):
        pivot = detail.pivot_table(index='ue_label', columns='phase', values=metric, aggfunc='mean').reindex(columns=['before', 'after_balanced'])
        pivot.plot(kind='bar', ax=ax, width=0.78)
        ax.set_ylabel(metric)
        ax.grid(axis='y', alpha=0.25)
    axes[-1].tick_params(axis='x', rotation=65)
    plt.tight_layout()
    moved_detail_plot_path = OUT_DIR / f'moved_ue_radio_before_after_{PER_UE_DETAIL_SCENARIO}.png'
    plt.savefig(moved_detail_plot_path, dpi=160, bbox_inches='tight')
    print('Saved:', moved_detail_plot_path)
    plt.show()

In [ ]:
# Check whether bad moved-UE SINR is just a timing artifact.
# This follows one scenario, records moved UEs immediately after the upper step,
# then neutralizes offsets and waits extra base-env radio steps.
WAIT_SENSITIVITY_SCENARIO = 'jain_balance_controllable'
WAIT_EXTRA_BASE_STEPS = [1, 5, 10, 20]

model = PPO.load(str(MODEL_PATH), device='cpu')
env_args = apply_reward_overrides(build_args(WAIT_SENSITIVITY_SCENARIO, SEED), training_reward_override_args())
env = make_env(env_args)
obs, reset_info = env.reset()
base_env = env.unwrapped.base_env

def simple_radio_rows(label):
    rows = []
    for ue in base_env.get_all_ues():
        m = base_env.get_ue_radio_metrics(int(ue.id))
        rows.append({
            'label': label,
            'ue_id': int(ue.id),
            'slice_type': getattr(ue, 'slice_type', ''),
            'serving_gnb': m.get('serving_gnb'),
            'x': m.get('x', np.nan),
            'y': m.get('y', np.nan),
            'sinr_db': m.get('sinr_db', np.nan),
            'scheduled_sinr_db': m.get('scheduled_sinr_db', np.nan),
            'throughput_mbps': m.get('throughput', np.nan) / 1e6,
            'delay_steps': m.get('delay_steps', np.nan),
            'queue_bits': m.get('queue', np.nan),
        })
    return rows

wait_rows = simple_radio_rows('before_action')
action, _ = model.predict(obs, deterministic=True)
obs, reward, terminated, truncated, info = env.step(action)
wait_rows.extend(simple_radio_rows('after_upper_step'))

before_serving = pd.DataFrame(wait_rows).query("label == 'before_action'").set_index('ue_id')['serving_gnb']
after_serving = pd.DataFrame(wait_rows).query("label == 'after_upper_step'").set_index('ue_id')['serving_gnb']
moved_ids = [int(ue_id) for ue_id in before_serving.index if before_serving.loc[ue_id] != after_serving.loc[ue_id]]

env.unwrapped._apply_slice_offsets(np.zeros_like(env.unwrapped._last_strong_offsets))
previous = 0
for total_steps in WAIT_EXTRA_BASE_STEPS:
    for _ in range(total_steps - previous):
        base_env.step(0)
    wait_rows.extend(simple_radio_rows(f'after_extra_{total_steps}_base_steps'))
    previous = total_steps
env.close()

wait_sensitivity = pd.DataFrame(wait_rows)
wait_sensitivity_moved = wait_sensitivity[wait_sensitivity['ue_id'].isin(moved_ids)].copy()
wait_sensitivity_path = OUT_DIR / f'wait_sensitivity_{WAIT_SENSITIVITY_SCENARIO}.csv'
wait_sensitivity_moved.to_csv(wait_sensitivity_path, index=False)
print('Saved:', wait_sensitivity_path)
print('Moved UE ids:', moved_ids)
display(wait_sensitivity_moved)

wait_plot = wait_sensitivity_moved.groupby('label', as_index=False).agg(
    sinr_db=('sinr_db', 'mean'),
    throughput_mbps=('throughput_mbps', 'mean'),
    delay_steps=('delay_steps', 'mean'),
)
wait_plot['label'] = pd.Categorical(wait_plot['label'], ['before_action', 'after_upper_step'] + [f'after_extra_{n}_base_steps' for n in WAIT_EXTRA_BASE_STEPS], ordered=True)
wait_plot = wait_plot.sort_values('label')

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, metric in zip(axes, ['sinr_db', 'throughput_mbps', 'delay_steps']):
    ax.plot(wait_plot['label'].astype(str), wait_plot[metric], marker='o')
    ax.set_ylabel(metric)
    ax.grid(alpha=0.25)
axes[-1].tick_params(axis='x', rotation=35)
plt.tight_layout()
wait_plot_path = OUT_DIR / f'wait_sensitivity_{WAIT_SENSITIVITY_SCENARIO}.png'
plt.savefig(wait_plot_path, dpi=160, bbox_inches='tight')
print('Saved:', wait_plot_path)
plt.show()

In [ ]:
artifacts = sorted(OUT_DIR.glob('*.csv')) + sorted(OUT_DIR.glob('*.json')) + sorted(OUT_DIR.glob('*.png'))
print('Artifacts:')
for path in artifacts:
    print('-', path)